# 03 · Agrupamiento y detección de anomalías

La idea de esta parte es buscar grupos de camiones que tengan un comportamiento parecido y también registros que se salgan de lo normal. Para entrenar usamos solo los sensores. Ya después miramos la etiqueta de falla para entender mejor los resultados.

Tomamos una muestra aleatoria de 10.000 registros del entrenamiento, sin tener en cuenta la clase. Así el análisis tarda menos, pero hay que tener presente que con otra muestra los resultados pueden cambiar y algunas fallas poco frecuentes pueden quedar por fuera. No usamos el conjunto de prueba.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score, davies_bouldin_score

raiz = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(raiz))
from src.data import load_aps
from src.preprocessing import build_preprocessor

figuras = raiz / "figures"
resultados = raiz / "results"
figuras.mkdir(exist_ok=True)
resultados.mkdir(exist_ok=True)
X, y = load_aps("train")
indices = X.sample(n=10000, random_state=42).index
datos = X.loc[indices].copy()
etiquetas = y.loc[indices].copy()
print("Muestra:", datos.shape)
print("Faltantes:", round(datos.isna().mean().mean() * 100, 2), "%")

Muestra: (10000, 170)
Faltantes: 8.32 %


## Preparación y métodos

Usamos el preprocesamiento que ya tenía el proyecto. Este quita columnas con más del 70 % de datos faltantes, rellena con la mediana y agrega indicadores de valores faltantes. También transforma los valores, elimina variables constantes o muy correlacionadas y escala los datos. Todo se ajusta con la muestra de entrenamiento.

Después usamos PCA para conservar al menos el 90 % de la varianza y reducir la cantidad de variables. Esto hace más manejable el cálculo, aunque también puede dejar por fuera información que sirva para distinguir algunas fallas.

Para buscar grupos usamos **K-Means**, que junta los registros según su distancia a un centro. Es fácil de interpretar, pero funciona mejor con grupos compactos y depende de la escala. Probamos entre 2 y 6 grupos y comparamos la silueta con una submuestra aleatoria.

También usamos **Isolation Forest** para buscar registros atípicos. Este método no necesita que los grupos tengan una forma determinada, pero el resultado depende de cuántas anomalías decidamos marcar. En este caso usamos el 2 % como referencia, no porque sepamos que esa sea la proporción real de fallas. Ninguno de los dos métodos usa la etiqueta al entrenar.

In [ ]:
preparador = build_preprocessor()
datos_preparados = preparador.fit_transform(datos)
pca = PCA(n_components=0.90, random_state=42)
datos_pca = pca.fit_transform(datos_preparados)
print("Variables después de preparar:", datos_preparados.shape[1])
print("Componentes PCA:", datos_pca.shape[1])
print("Varianza conservada:", round(pca.explained_variance_ratio_.sum(), 3))

Variables después de preparar: 141
Componentes PCA: 53
Varianza conservada: 0.903


In [ ]:
comparacion = []
for cantidad in range(2, 7):
    modelo = KMeans(n_clusters=cantidad, random_state=42, n_init=10)
    grupos = modelo.fit_predict(datos_pca)
    silueta = silhouette_score(datos_pca, grupos, sample_size=2000, random_state=42)
    davies = davies_bouldin_score(datos_pca, grupos)
    comparacion.append([cantidad, silueta, davies])

comparacion = pd.DataFrame(comparacion, columns=["grupos", "silueta", "davies_bouldin"])
comparacion.to_csv(resultados / "03_comparacion_grupos.csv", index=False)
print(comparacion.round(3).to_string(index=False))
cantidad_elegida = int(comparacion.loc[comparacion["silueta"].idxmax(), "grupos"])
kmeans = KMeans(n_clusters=cantidad_elegida, random_state=42, n_init=10)
grupo = kmeans.fit_predict(datos_pca)
aislamiento = IsolationForest(contamination=0.02, random_state=42, n_jobs=-1)
anomalia = aislamiento.fit_predict(datos_pca) == -1
print("Grupos elegidos:", cantidad_elegida)
print("Anomalías:", anomalia.sum())

 grupos  silueta  davies_bouldin
      2    0.298           1.362
      3    0.294           2.065
      4    0.301           1.780
      5    0.245           1.827
      6    0.251           1.940
Grupos elegidos: 4
Anomalías: 200


## Interpretación con la etiqueta

Ahora sí miramos cuántas fallas hay en cada grupo y entre las anomalías. Ojo, que un registro sea atípico no significa necesariamente que tenga una falla APS. También comparamos algunas de las variables que fueron importantes en el análisis supervisado. Como los nombres de los sensores son anónimos, no podemos decir qué mide cada uno.

In [ ]:
analisis = datos.copy()
analisis["grupo"] = grupo
analisis["anomalia"] = anomalia
analisis["falla"] = etiquetas.to_numpy()
perfiles = analisis.groupby("grupo").agg(registros=("falla", "size"), fallas=("falla", "sum"), tasa_fallas=("falla", "mean"), anomalias=("anomalia", "sum"))
perfiles.to_csv(resultados / "03_perfiles_grupos.csv")
print(perfiles.round(4).to_string())
resumen_anomalias = analisis.groupby("anomalia").agg(registros=("falla", "size"), fallas=("falla", "sum"), tasa_fallas=("falla", "mean"))
resumen_anomalias.to_csv(resultados / "03_anomalias.csv")
print("\nAnomalías:")
print(resumen_anomalias.round(4).to_string())
variables = ["aa_000", "ck_000", "bs_000", "cb_000", "ai_000"]
medianas = analisis.groupby("grupo")[variables].median()
medianas.to_csv(resultados / "03_medianas_grupos.csv")
print("\nMedianas por grupo:")
print(medianas.round(2).to_string())
print("\nMedianas por condición de anomalía:")
print(analisis.groupby("anomalia")[variables].median().round(2).to_string())

       registros  fallas  tasa_fallas  anomalias
grupo                                           
0           3836       8       0.0021         27
1           5008      37       0.0074         41
2             87       3       0.0345         19
3           1069     122       0.1141        113

Anomalías:
          registros  fallas  tasa_fallas
anomalia                                
False          9800     118        0.012
True            200      52        0.260

Medianas por grupo:
        aa_000      ck_000    bs_000    cb_000  ai_000
grupo                                                 
0        262.0     9765.12   17380.0   67340.0     0.0
1      40782.0   406162.08   89250.0  575550.0     0.0
2       4040.0    46987.68   38540.0  136240.0     0.0
3      94582.0  1009090.56  121480.0  599870.0     0.0

Medianas por condición de anomalía:
            aa_000      ck_000    bs_000    cb_000  ai_000
anomalia                                                  
False      30584.0   245

In [ ]:
figura, ejes = plt.subplots(1, 2, figsize=(13, 5))
ejes[0].scatter(datos_pca[:, 0], datos_pca[:, 1], c=grupo, s=5, alpha=0.5, cmap="tab10")
ejes[0].set_title("Grupos de K-Means")
ejes[1].scatter(datos_pca[~anomalia, 0], datos_pca[~anomalia, 1], s=4, alpha=0.2, label="Otros")
ejes[1].scatter(datos_pca[anomalia, 0], datos_pca[anomalia, 1], s=8, alpha=0.7, label="Anomalías")
ejes[1].set_title("Isolation Forest")
ejes[1].legend()
for eje in ejes:
    eje.set_xlabel("Componente principal 1")
    eje.set_ylabel("Componente principal 2")
figura.tight_layout()
figura.savefig(figuras / "03_grupos_anomalias.png", dpi=130, bbox_inches="tight")
plt.show()

## Limitaciones

Las métricas nos ayudan a ver cómo quedaron separados los grupos en PCA, pero por sí solas no dicen si esos grupos sirven para mantenimiento. Además, la gráfica solo muestra dos componentes, mientras que los modelos usaron más información.

Estos resultados describen la muestra que usamos. No prueban que una variable cause una falla ni aseguran que vaya a pasar lo mismo con otros datos. Sería bueno repetir el análisis con otras muestras y revisar los perfiles con personas que conozcan el funcionamiento de los camiones.

## Resultados de esta ejecución

De las opciones entre 2 y 6 grupos, elegimos 4 porque dieron la silueta más alta, con 0,301. El índice Davies-Bouldin fue 1,780. El grupo 3 tiene 1.069 registros, de los cuales 122 son fallas: un 11,41 %, frente al 1,70 % de toda la muestra. En ese grupo las medianas de `aa_000` y `ck_000` también son más altas que en los otros grupos. Esto muestra una diferencia en los sensores, aunque no sabemos qué significa físicamente cada variable.

Isolation Forest marcó 200 registros como anomalías y 52 de ellos tenían falla, o sea el 26 %. En los 9.800 registros que no marcó hubo 118 fallas, aproximadamente el 1,20 %. El método encontró 52 de las 170 fallas de la muestra, así que no reemplaza al clasificador supervisado. Además, 113 de las 200 anomalías quedaron en el grupo 3. Hay una concentración de fallas y cierta coincidencia entre los métodos, pero no todos los registros atípicos son fallas.

![Grupos y anomalías](../figures/03_grupos_anomalias.png)